# Figure 3: Baseline-centered event trajectories

Display each treated home's baseline-centered daily trajectory and the equal-home eligible comparison mean around seven retrofit events. Seven-day smoothing is for display only and is not an input to DiD or CITS.

**Step:** 14. Article figure. See README.md for its data prerequisites.

**Article scope:** the main article and Supplementary Material.

**Inputs:** Read trajectory_plot_data.csv, events.csv, and event_results.csv from the selected analysis directory. Run 11_result_tables.ipynb first to use newly assembled trajectories.

**Outputs:** The named figure in the separate reproduction figures directory, using the original export formats.

**How to use this notebook:** select the Python environment described in `README.md`, restart the kernel,
and run the cells from top to bottom. Edit the configuration cell before running.
The checks below stop execution when the current data no longer match the manuscript specification.

## Locate the code folder and load the shared functions

The repository is found by searching this notebook's folder and its parent folders, so Jupyter can be started anywhere inside the repository. Set `INDOOR_AQ_CODE` only when it starts elsewhere.
It loads only the cells tagged `library` in `00_shared_functions.ipynb` (its demonstration cell is not executed).
Each notebook has its own imports and configuration, so it never depends on another notebook's kernel state.

In [ ]:
# Setup: find the code folder and load the shared functions.
from pathlib import Path
import json
import os
import sys
import types


def locate_code_directory():
    """Locate this repository (the folder with 00_shared_functions.ipynb) from the working directory or INDOOR_AQ_CODE."""
    configured = os.environ.get("INDOOR_AQ_CODE")
    if configured:
        candidates = [Path(configured).expanduser().resolve()]
    else:
        candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if ((candidate / "00_shared_functions.ipynb").is_file()
                and (candidate / "5_did_cits").is_dir()):
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter in the repository folder, or set INDOOR_AQ_CODE to that folder.")

CODE = locate_code_directory()

# Load the cells tagged "library" from 00_shared_functions.ipynb as an in-memory
# module named `shared`, so that `from shared import ...` works below.
# No helper .py file is generated.
_library_path = CODE / "00_shared_functions.ipynb"
_library = types.ModuleType("shared")
_library.__file__ = str(_library_path)
_library.__dict__["CODE"] = CODE
for _cell in json.loads(_library_path.read_text(encoding="utf-8"))["cells"]:
    if _cell["cell_type"] == "code" and "library" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), str(_library_path), "exec"), _library.__dict__)
sys.modules["shared"] = _library

from shared import DATA, output_root
print(f"Manuscript code: {CODE}")

## Configure figure paths and resolution

Only presentation and output settings belong here. Keep the manuscript data and event specification fixed when reproducing the final figure.

In [ ]:
# Paths are shared by Python and R. Defaults: data/ for supplied inputs,
# results/ for this run. Set INDOOR_AQ_DATA and INDOOR_AQ_OUTPUT before
# starting Jupyter and R; relative paths are resolved against the repository.
OUTPUT = os.environ.get("INDOOR_AQ_OUTPUT") or None

# Set a lower DPI only for an explicit draft; manuscript exports use 900.
FIGURE_DPI = int(os.environ.get("INDOOR_AQ_FIGURE_DPI", "900"))
os.environ["INDOOR_AQ_OUTPUT"] = str(output_root(OUTPUT))
os.environ["INDOOR_AQ_FIGURE_DPI"] = str(FIGURE_DPI)
# All event-result figures use this run's results folder.


## Load plotting libraries and define the figure settings

The following constants preserve the current figure's outcome order, labels, colors, panel geometry, and reference lines. Numerical methodology and interpretation are documented in the preceding and following cells.

In [ ]:
import os
import sys
from pathlib import Path
from shared import CODE, DATA, output_root, load_hourly as load_manuscript_hourly

import warnings

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ---- Input data ------------------------------------------------------------
ANALYSIS_DIR = output_root()
TRAJECTORY_PATH = ANALYSIS_DIR / 'tables' / 'trajectory_plot_data.csv'
EVENTS_PATH = ANALYSIS_DIR / 'data' / 'events.csv'
RESULTS_PATH = ANALYSIS_DIR / 'tables' / 'event_results.csv'

# ---- Output ----------------------------------------------------------------
OUTPUT_DIR = output_root() / 'figures'
OUTPUT_STEM = 'Fig3_event_trajectories'
EXPORT_FORMATS = ('png', 'pdf')
DPI = int(os.environ.get('INDOOR_AQ_FIGURE_DPI', '900'))

# ---- Font and library version used for the manuscript figure ----------------
FONT_FAMILY = 'Liberation Sans'
FALLBACK_FONT = 'Arial'  # same character widths as Liberation Sans, so the layout is unchanged
# Folders searched for LiberationSans-*.ttf if the font is not installed.
FONT_SEARCH_DIRS = [
    Path(os.environ.get('WINDIR', r'C:\Windows')) / 'Fonts',
    Path(os.environ.get('LOCALAPPDATA', '')) / 'Microsoft' / 'Windows' / 'Fonts',
    Path(r'C:\Program Files\LibreOffice\share\fonts\truetype'),
    # Optional: a folder with LiberationSans-*.ttf (e.g. unzipped from the
    # Liberation fonts release) can be supplied through INDOOR_AQ_FONT_DIR.
    *([Path(os.environ['INDOOR_AQ_FONT_DIR'])] if os.environ.get('INDOOR_AQ_FONT_DIR') else []),
]
EXPECTED_MATPLOTLIB = '3.10'  # other releases may shift text by a sub-pixel

# ---- Rows: retrofit events, top to bottom -----------------------------------
# Display labels use 'dMEV' (UK government usage; 'DMEV' before 12 Oct 2026).
# Event codes such as 'H05_DMEV' are identifiers in the saved results and are unchanged.
EVENTS = ['H05_Loft', 'H05_DMEV', 'H10_Loft', 'H10_DMEV', 'H14_CWI', 'H14_Loft', 'H16_Bundle']
EVENT_LABELS = {
    'H05_Loft': ('Home 5', 'Loft insulation'),
    'H05_DMEV': ('Home 5', 'dMEV'),
    'H10_Loft': ('Home 10', 'Loft insulation'),
    'H10_DMEV': ('Home 10', 'dMEV'),
    'H14_CWI': ('Home 14', 'CWI'),
    'H14_Loft': ('Home 14', 'Loft insulation'),
    'H16_Bundle': ('Home 16', 'Combined works'),
}

# ---- Columns: outcomes ---------------------------------------------------------
OUTCOMES = ['temp', 'rh', 'pm25', 'co2']
OUTCOME_LABELS = {
    'temp': 'Temperature (°C)',
    'rh': 'RH (percentage points)',
    'pm25': 'PM$_{2.5}$ (µg/m$^3$)',
    'co2': 'CO$_2$ (ppm)',
}
Y_TICKS = {
    'temp': [-5, 0, 5],
    'rh': [-10, 0, 10],
    'pm25': [-25, 0, 25, 50],
    'co2': [-200, -100, 0, 100],
}
Y_PADDING = 0.06                 # fraction of the data range added above and below
RH_MINIMUM_LIMITS = (-10.8, 11.5)  # RH limits widened so the ±10 ticks fit

# ---- Colors and line styles ------------------------------------------------
TREATED_COLOR = '#7A2E5A'
COMPARISON_COLOR = '#6E6E6E'
COMPARISON_DASH = (0, (3, 1.5))
CONSTRUCTION_COLOR = '#D9D9D9'
ZERO_LINE_COLOR = '#BDBDBD'
COUNT_TEXT_COLOR = '#404040'

# Series drawn in this order: (series name in the data, color, line style, width, z-order)
SERIES_STYLES = [
    ('Eligible comparisons', COMPARISON_COLOR, COMPARISON_DASH, 1.0, 2),
    ('Treated home', TREATED_COLOR, '-', 1.2, 3),
]

# ---- Figure layout (inches and figure fractions) -----------------------------
FIGURE_SIZE = (7.48, 9.6)
SUBPLOT_LAYOUT = dict(left=0.165, right=0.985, top=0.965, bottom=0.055, hspace=0.55, wspace=0.30)
LEGEND_ANCHOR = (0.56, 0.0)

## Check matplotlib version

Report whether Matplotlib matches the release used for the manuscript. Fonts and text placement may differ across releases even when the underlying data and geometry agree.

In [ ]:
def check_matplotlib_version(expected=EXPECTED_MATPLOTLIB):
    """Warn if matplotlib differs from the release used for the manuscript figure.

    The plotted data are unaffected, but text can be placed a sub-pixel
    differently between releases, so the PNG would not be pixel-identical.
    """
    if not matplotlib.__version__.startswith(expected + '.'):
        warnings.warn(
            f'The manuscript figure was rendered with matplotlib {expected}.x; '
            f'this kernel has {matplotlib.__version__}.')

## Resolve font

Prefer the manuscript font, register an available local font file if needed, and use the stated fallback when unavailable. This affects rendering, not data selection or effect estimation.

In [ ]:
def resolve_font(family=FONT_FAMILY, search_dirs=FONT_SEARCH_DIRS, fallback=FALLBACK_FONT):
    """Return a font family that matplotlib can use, preferring ``family``.

    If ``family`` is not installed, any matching TrueType files found in
    ``search_dirs`` (e.g. LiberationSans-Regular.ttf) are registered for this
    session. If none are found, ``fallback`` is returned with a warning.
    """
    def available():
        return {font.name for font in font_manager.fontManager.ttflist}

    if family in available():
        return family

    pattern = family.replace(' ', '') + '-*.ttf'
    for directory in search_dirs:
        for path in sorted(Path(directory).glob(pattern)):
            font_manager.fontManager.addfont(str(path))

    if family in available():
        return family
    warnings.warn(f'{family} not found; using {fallback}. Glyph shapes will differ slightly.')
    return fallback

## Apply style

Reset Matplotlib defaults before applying the figure's font, line, and export settings. This prevents unrelated notebook styling from carrying over into the manuscript figure.

In [ ]:
def apply_style(font_family):
    """Reset matplotlib to its defaults, then apply the Fig. 3 style.

    Resetting first keeps the output independent of earlier notebook cells
    and of any personal matplotlibrc file.
    """
    plt.rcdefaults()
    plt.rcParams.update({
        'font.family': font_family,
        'font.size': 8,
        'axes.linewidth': 0.6,
        'xtick.major.width': 0.6,
        'ytick.major.width': 0.6,
        'xtick.major.size': 2.5,
        'ytick.major.size': 2.5,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'mathtext.default': 'regular',  # sub/superscripts in the text font, not italic math
        'savefig.facecolor': 'white',
        'pdf.fonttype': 42,             # embed TrueType fonts so PDF text stays editable
    })

## Load trajectories

Read the centered and smoothed trajectory table assembled by 11_result_tables.ipynb. Require the expected series and parse dates before plotting; the unsmoothed daily outcome remains the model input.

In [ ]:
def load_trajectories(path=TRAJECTORY_PATH):
    """Load the daily trajectories plotted in each panel.

    Returns
    -------
    pandas.DataFrame
        Columns ``event``, ``outcome``, ``series`` ('Treated home' or
        'Eligible comparisons'), ``date`` and ``smoothed_value`` (centered
        seven-day mean of the baseline-centered daily mean; NaN where fewer
        than three daily values were available).
    """
    return pd.read_csv(path, parse_dates=['date'])

## Load events

Read work-start and work-end dates and retain the fixed event order. Construction shading and date labels use these same dates.

In [ ]:
def load_events(path=EVENTS_PATH):
    """Load work start and end dates, indexed by event code."""
    return pd.read_csv(path, parse_dates=['start', 'end']).set_index('event')

## Load comparison counts

Read outcome-specific comparison counts from the primary result rows. PM2.5 can have fewer comparisons because of the baseline concentration rule.

In [ ]:
def load_comparison_counts(path=RESULTS_PATH):
    """Number of comparison homes per (event, outcome) in the primary analysis."""
    results = pd.read_csv(path)
    primary = results[results['analysis'].eq('primary')]
    return primary.groupby(['event', 'outcome'])['n_controls'].first()

## Shared y limits

Choose a common vertical range for each outcome across all seven events. Shared scales make event panels comparable without rescaling each event separately.

In [ ]:
def shared_y_limits(trajectories):
    """Y-axis limits shared by every panel in a column.

    Each outcome uses the full range of its smoothed values across all
    events, padded by ``Y_PADDING`` of the range on both sides. RH is widened
    to at least ``RH_MINIMUM_LIMITS``.
    """
    limits = {}
    for outcome in OUTCOMES:
        values = trajectories.loc[trajectories['outcome'].eq(outcome), 'smoothed_value'].dropna()
        low, high = values.min(), values.max()
        pad = Y_PADDING * (high - low)
        limits[outcome] = (low - pad, high + pad)

    low, high = limits['rh']
    limits['rh'] = (min(low, RH_MINIMUM_LIMITS[0]), max(high, RH_MINIMUM_LIMITS[1]))
    return limits

## Day month

Format a compact day-month label for the original figure layout. Date formatting does not change the underlying calendar time.

In [ ]:
def day_month(date):
    """'30 May' style label."""
    return f'{date.day} {date.month_name()[:3]}'

## Day month year

Include the year in an event date label where the row annotation requires it.

In [ ]:
def day_month_year(date):
    """'30 May 2024' style label."""
    return f'{day_month(date)} {date.year}'

## Work dates label

Format a single-day intervention or a multi-day work interval using the exact start and end dates.

In [ ]:
def work_dates_label(start, end):
    """Work date(s): a single date, or a 'start–end' range for multi-day works."""
    if start == end:
        return day_month_year(start)
    return f'{day_month(start)}–{day_month_year(end)}'

## Draw trajectories

Draw treated and comparison series on their actual calendar dates, shade the full construction interval, and retain gaps. Lines show baseline-centered display values, not treatment-effect estimates.

In [ ]:
def draw_trajectories(ax, panel, start, end):
    """Draw the comparison mean, the treated home and the construction band.

    Each series is reindexed to every calendar day in the window so that
    missing dates appear as gaps rather than being bridged by the line.

    Returns
    -------
    first, last : pandas.Timestamp
        First and last dates of the panel window.
    """
    first, last = panel['date'].min(), panel['date'].max()
    all_days = pd.date_range(first, last)

    for series, color, linestyle, linewidth, zorder in SERIES_STYLES:
        values = (panel[panel['series'].eq(series)]
                  .set_index('date')['smoothed_value']
                  .reindex(all_days))
        ax.plot(values.index, values.values, color=color, ls=linestyle, lw=linewidth, zorder=zorder)

    ax.axhline(0, color=ZERO_LINE_COLOR, lw=0.5, zorder=1)
    # Half a day either side so the band covers the whole of each work day
    ax.axvspan(start - pd.Timedelta(hours=12), end + pd.Timedelta(hours=12),
               color=CONSTRUCTION_COLOR, lw=0, zorder=0)
    return first, last

## Format date axis

Use the original baseline and follow-up date positions and concise labels for each event panel.

In [ ]:
def format_date_axis(ax, first, start, last):
    """Label the window start, the work start and the window end on the x axis."""
    ax.set_xlim(first, last)
    ax.set_xticks([first, start, last])
    ax.set_xticklabels([day_month(first), day_month(start), day_month(last)])

    # Keep the end labels inside the panel. The work-start label is
    # right-aligned when it sits close to the window end, to avoid overlap.
    labels = ax.get_xticklabels()
    labels[0].set_ha('left')
    labels[2].set_ha('right')
    labels[1].set_ha('right' if (last - start).days < 45 else 'center')
    ax.tick_params(axis='x', labelsize=6.3, pad=1.5)

## Format value axis

Apply the shared outcome limits, reference zero line, and fixed tick labels. The outcome units stay consistent across rows.

In [ ]:
def format_value_axis(ax, outcome, limits):
    """Apply the column's shared y limits and fixed y ticks."""
    ax.tick_params(axis='y', labelsize=6.5, pad=1.5)
    ax.set_ylim(*limits[outcome])
    ax.set_yticks(Y_TICKS[outcome])

## Add comparison count

Annotate the eligible comparison count for that event and outcome, which is taken from the fitted primary result table.

In [ ]:
def add_comparison_count(ax, n_comparisons):
    """Print the number of comparison homes above the top-right of the panel."""
    ax.text(1.0, 1.03, f'n = {n_comparisons}', transform=ax.transAxes,
            ha='right', va='bottom', fontsize=6.2, color=COUNT_TEXT_COLOR)

## Add row label

Identify the treated home, retrofit event, and work dates without adding analysis settings inside the plotting area.

In [ ]:
def add_row_label(ax, row, event, start, end):
    """Write '(a) Home 5 / Loft insulation / 30 May 2024' to the left of a row."""
    home, measure = EVENT_LABELS[event]
    text = f'({chr(ord("a") + row)}) {home}\n{measure}\n{work_dates_label(start, end)}'
    ax.annotate(text, xy=(-0.30, 0.5), xycoords='axes fraction',
                ha='right', va='center', fontsize=7.2, linespacing=1.3)

## Add legend

Explain the series or model markers once in a shared legend. The labels match the manuscript's displayed groups.

In [ ]:
def add_legend(fig):
    """Add the shared legend below the grid."""
    handles = [
        Line2D([], [], color=TREATED_COLOR, lw=1.3, label='Treated home'),
        Line2D([], [], color=COMPARISON_COLOR, lw=1.1, ls=COMPARISON_DASH,
               label='Mean of eligible comparison homes'),
        Patch(color=CONSTRUCTION_COLOR, label='Construction'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False,
               fontsize=7.5, bbox_to_anchor=LEGEND_ANCHOR)

## Build figure

Assemble the complete manuscript panel layout from the validated inputs and drawing helpers. This function contains presentation logic and does not fit a statistical model.

In [ ]:
def build_figure(trajectories, events, comparison_counts):
    """Build the 7 x 4 grid of Fig. 3 and return the figure.

    Parameters
    ----------
    trajectories : pandas.DataFrame
        Output of :func:`load_trajectories`.
    events : pandas.DataFrame
        Output of :func:`load_events`.
    comparison_counts : pandas.Series
        Output of :func:`load_comparison_counts`.
    """
    fig, axes = plt.subplots(len(EVENTS), len(OUTCOMES), figsize=FIGURE_SIZE, sharey='col')
    limits = shared_y_limits(trajectories)

    for row, event in enumerate(EVENTS):
        start, end = events.loc[event, 'start'], events.loc[event, 'end']
        for col, outcome in enumerate(OUTCOMES):
            ax = axes[row, col]
            panel = trajectories[trajectories['event'].eq(event)
                                 & trajectories['outcome'].eq(outcome)]
            first, last = draw_trajectories(ax, panel, start, end)
            format_date_axis(ax, first, start, last)
            format_value_axis(ax, outcome, limits)
            add_comparison_count(ax, comparison_counts[(event, outcome)])
            if row == 0:
                ax.set_title(OUTCOME_LABELS[outcome], fontsize=8.5, pad=12)
        add_row_label(axes[row, 0], row, event, start, end)

    add_legend(fig)
    fig.subplots_adjust(**SUBPLOT_LAYOUT)
    return fig

## Save figure

Export the figure to the original file formats. PNG uses the configured resolution (900 dpi by default); PDF and SVG, when included, preserve vector elements. The output directory is separate from the manuscript source figures.

In [ ]:
def save_figure(fig, output_dir=OUTPUT_DIR, stem=OUTPUT_STEM,
                formats=EXPORT_FORMATS, dpi=DPI):
    """Save ``fig`` once per format and return the written paths."""
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for ext in formats:
        path = output_dir / f'{stem}.{ext}'
        fig.savefig(path, dpi=dpi)
        paths.append(path)
    return paths

## Build and export the figure

Read trajectory_plot_data.csv, events.csv, and event_results.csv from the selected analysis directory. Run 11_result_tables.ipynb first to use newly assembled trajectories. The cells above expose each processing and plotting step for inspection. The following execution writes the figure and prints its actual paths.

In [ ]:
check_matplotlib_version()
apply_style(resolve_font())

trajectories = load_trajectories()
events = load_events()
comparison_counts = load_comparison_counts()

fig = build_figure(trajectories, events, comparison_counts)
for path in save_figure(fig):
    print(f'Saved {path}')

## Display a compact notebook preview

The preview uses a smaller PNG rendered from the same figure object. The saved publication PNG remains at the configured 900 dpi. Keep methodological detail in the caption when preparing the manuscript.

In [ ]:
import io
from IPython.display import Image, display
_preview = io.BytesIO()
fig.savefig(_preview, format="png", dpi=110, bbox_inches="tight")
display(Image(data=_preview.getvalue()))
plt.close(fig)